# 5.2 소프트 마진: 완벽히 나뉘지 않는 데이터 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter05_2_soft_margin.ipynb)

책 본문: [5.2 소프트 마진](https://smhanlab.com/book-ml/kor/ml1/chapter05/2.html)

이 노트북은 책 5.2절의 내용을 코드로 재현합니다: (1) 본문 "손으로 한 번"의
힌지 손실 표를 코드로 검증하고, (2) 소프트 마진의 **세 가지 영역**(마진 바깥 /
마진 안 / 옳은 쪽이지만 경계 너머)을 그림으로 보고, (3) 이상치 하나가 들어간
데이터에서 **$C$를 바꾸며** 결정 경계·마진·서포트 벡터가 어떻게 달라지는지
스윕하고, (4) **라벨 노이즈**가 섞인 데이터에서 $C$가 과적합을 어떻게
조절하는지(로지스틱회귀와 대조) 확인하고, (5) 힌지 손실과 교차 엔트로피의
**점별 그래디언트 기여**를 비교합니다. numpy/scikit-learn만 씁니다.


## 1. 본문 "손으로 한 번" 표 검증: 힌지 손실과 $C$의 효과

본문 설정: 후보 경계 $w=(0.2, 0.2),\ b=0$, 점 $(3,3){+1}, (-3,-3){-1}$은
마진을 만족하고, $(2,-3){+1}, (-2,3){-1}$은 반대쪽 영역에 놓인 이상치.
각 점의 $y^{(i)}(w^Tx^{(i)}+b)$와 힌지 손실 $\max(0, 1-y^{(i)}(w^Tx^{(i)}+b))$을
코드로 다시 계산한다.


In [1]:
import numpy as np

w, b = np.array([0.2, 0.2]), 0.0
pts = [(np.array([3, 3]), 1, "(3,3)"), (np.array([-3, -3]), -1, "(-3,-3)"),
       (np.array([2, -3]), 1, "(2,-3)"), (np.array([-2, 3]), -1, "(-2,3)")]
print(f"{'점':12s} {'y*(wTx+b)':>10s}  {'힌지 손실':>10s}")
tot = 0.0
for x, y, name in pts:
    m = y * (w @ x + b)
    h = max(0.0, 1.0 - m)
    tot += h
    print(f"{name:12s} {m:10.2f}  {h:10.2f}")
reg = 0.5 * w @ w
print(f"\nsum xi = {tot:.2f},   0.5||w||^2 = {reg:.4f}")
for C in (0.1, 10.0):
    print(f"  C={C:<5g}  J = {reg} + {C}*{tot} = {reg + C*tot:.2f}")


점             y*(wTx+b)       힌지 손실
(3,3)              1.20        0.00
(-3,-3)            1.20        0.00
(2,-3)            -0.20        1.20
(-2,3)            -0.20        1.20

sum xi = 2.40,   0.5||w||^2 = 0.0400
  C=0.1    J = 0.04000000000000001 + 0.1*2.4 = 0.28
  C=10     J = 0.04000000000000001 + 10.0*2.4 = 24.04


## 2. 소프트 마진의 기하: 세 가지 영역

5.1절의 하드 마진에서는 마진 경계 $f=\pm1$ 사이에 아무 데이터도 없었다.
소프트 마진에서는 데이터가 세 영역에 나뉜다:

1. **마진 바깥 정답** ($y(w^Tx+b) \ge 1$): 손실 0, 서포트 벡터 아님.
2. **마진 안 정답** ($0 < y(w^Tx+b) < 1$): $\xi^{(i)} = 1 - y(w^Tx+b)$만큼
   손실. 서포트 벡터(0 < $\alpha_i$ < $C$).
3. **옳은 쪽이지만 마진 너머** ($y(w^Tx+b) \le 0$): 손실 $\ge 1$.
   서포트 벡터($\alpha_i = C$, 이상치처럼 취급됨).

5.1절 데이터(대각선 두 클러스터)에 이상치 두 개 $(2,-3){+1}, (-2,3){-1}$를
추가하고 $C=1.0$으로 학습시키면, 이상치가 어느 영역에 빠지게 되는지
한눈에 확인된다.


In [2]:
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
from sklearn.svm import SVC

# 5.1절 데이터 + 이상치 두 개
rng = np.random.default_rng(7)
X = np.vstack([rng.normal([5, 5], 0.8, (40, 2)),
               rng.normal([-5, -5], 0.8, (40, 2)),
               np.array([[2.0, -3.0], [-2.0, 3.0]])])
y = np.r_[np.ones(40), -np.ones(40), [1.0, -1.0]]

clf = SVC(kernel="linear", C=1.0).fit(X, y)
wc, bc = clf.coef_[0], clf.intercept_[0]
sv = X[clf.support_]
out = X[80:82]                       # (2,-3), (-2,3)
x1 = np.linspace(-8, 8, 100)
f_line = lambda c: (-wc[0] * x1 - bc + c) / wc[1]

fig, ax = plt.subplots(figsize=(6.5, 6.5))
for lbl, m in [(1, "red"), (-1, "blue")]:
    ax.scatter(X[y == lbl, 0], X[y == lbl, 1], color=m, alpha=0.6, label=f"y={lbl}")
ax.plot(x1, f_line(0),  "k",   lw=2,  label="결정 경계 (f=0)")
ax.plot(x1, f_line(1),  "g--", lw=1,  label="마진 경계 (f=+1)")
ax.plot(x1, f_line(-1), "g--", lw=1,  label="마진 경계 (f=-1)")
ax.scatter(sv[:, 0], sv[:, 1], s=160, facecolors="none",
           edgecolors="orange", lw=2.5, label="서포트 벡터")
ax.scatter(out[:, 0], out[:, 1], s=260, marker="*",
           facecolors="white", edgecolors="red", lw=1.5, label="이상치")
ax.set_aspect("equal"); ax.legend(loc="upper left", fontsize=9)
ax.set_title("소프트 마진: 이상치가 마진 안/너머로 허용됨 (C=1)")
fig.tight_layout()
fig.savefig("/home/smhan/book-ml/kor/src/images/ch05_2_soft_margin.svg")
plt.show()

for i in [80, 81]:
    m = y[i] * (wc @ X[i] + bc)
    print(f"이상치 {tuple(X[i])}: y(wTx+b) = {m:+.3f}  ->  xi = {max(0.0, 1-m):.3f}, "
          f"{'서포트 벡터' if i in clf.support_ else '아님'}")
print(f"서포트 벡터 {len(sv)}개 / 전체 {len(X)}개,  마진 2/||w|| = {2/np.linalg.norm(wc):.3f}")


이상치 (np.float64(2.0), np.float64(-3.0)): y(wTx+b) = +1.000  ->  xi = 0.000, 서포트 벡터
이상치 (np.float64(-2.0), np.float64(3.0)): y(wTx+b) = +1.000  ->  xi = 0.000, 서포트 벡터
서포트 벡터 3개 / 전체 82개,  마진 2/||w|| = 4.743


## 3. $C$ 스윕: 이상치 하나, 경계의 "굽힘" 조절

이상치가 **정확히 어느 쪽 마진 안**인지에 따라 $C$의 효과가 달라진다.
여기서는 클러스터 $\pm(2.5,2.5)$ (스프레드 0.6, 40+40개)에
$y{=}{+}1$인 이상치 $(-1.6, 0.8)$ 하나를 놓는다 — $(-2.5,-2.5)$ 클러스터의
여유 영역에 들어와 "거의" 옳은 쪽에 있지만 마진은 위반하는 점이다.
$C$를 $0.01$부터 $100$까지 올리면:

- $C$가 작으면: 이상치를 아예 무시하고 **넓은 마진**을 유지
- $C$가 크면: 이상치를 제자리(최소 $y(w^Tx+b)\ge1$)로 끌어오려다
  **마진이 좁아지고** 경계가 기울어짐


In [3]:
rng = np.random.default_rng(21)
n = 40
X = np.vstack([rng.normal([2.5, 2.5], 0.6, (n, 2)),
               rng.normal([-2.5, -2.5], 0.6, (n, 2)),
               np.array([[-1.6, 0.8]])])
y = np.r_[np.ones(n), -np.ones(n), [1.0]]
print(f"{'C':>7s} {'마진 2/||w||':>12s} {'b/||w||':>8s} {'n_SV':>5s} {'이상치 y(wTx+b)':>15s} {'정상점 오분류':>12s}")
for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
    clf = SVC(kernel="linear", C=C).fit(X, y)
    wc, bc = clf.coef_[0], clf.intercept_[0]
    om = y[-1] * (wc @ X[-1] + bc)
    mis = (clf.predict(X[:-1]) != y[:-1]).sum()
    print(f"{C:7.2f} {2/np.linalg.norm(wc):12.3f} {bc/np.linalg.norm(wc):+8.3f} "
          f"{len(clf.support_):5d} {om:15.3f} {mis:12d}")


      C   마진 2/||w||  b/||w||  n_SV    이상치 y(wTx+b)      정상점 오분류
   0.01        6.006   +0.018    14          -0.160            0
   0.10        4.612   +0.230     5           0.075            0
   1.00        2.298   +0.650     2           1.000            0
  10.00        2.298   +0.650     2           1.000            0
 100.00        2.298   +0.650     2           1.000            0


In [4]:
# 같은 데이터를 C = 0.01, 0.1, 1, 100 로 학습시킨 결정 경계를 한 그림에
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Cs = [0.01, 0.1, 1.0, 100.0]
colors = ["#1a53ff", "#40c4ff", "#ffa500", "#ff1744"]   # 청 -> 적 (C 증대)
fig, axes = plt.subplots(2, 2, figsize=(10, 9.5))
x1 = np.linspace(-6, 6, 100)
for ax, C, col in zip(axes.ravel(), Cs, colors):
    clf = SVC(kernel="linear", C=C).fit(X, y)
    wc, bc = clf.coef_[0], clf.intercept_[0]
    sv = X[clf.support_]
    fl = lambda c: (-wc[0] * x1 - bc + c) / wc[1]
    for lbl, m in [(1, "red"), (-1, "blue")]:
        ax.scatter(X[y == lbl, 0], X[y == lbl, 1], color=m, alpha=0.5, s=24)
    ax.scatter(X[-1, 0], X[-1, 1], s=320, marker="*",
               facecolors="white", edgecolors="red", lw=1.5, zorder=5)
    ax.plot(x1, fl(0),  color=col, lw=2.5, label=f"결정 경계")
    ax.plot(x1, fl(1),  color=col, lw=1, ls="--", alpha=0.6)
    ax.plot(x1, fl(-1), color=col, lw=1, ls="--", alpha=0.6)
    ax.scatter(sv[:, 0], sv[:, 1], s=130, facecolors="none",
               edgecolors="orange", lw=2, zorder=4)
    om = y[-1] * (wc @ X[-1] + bc)
    ax.set_title(f"C={C}:  마진={2/np.linalg.norm(wc):.2f}, "
                 f"n_SV={len(clf.support_)}, 이상치 y(wTx+b)={om:+.2f}", fontsize=11)
    ax.set_aspect("equal"); ax.legend(fontsize=8, loc="upper left")
fig.suptitle("C 스윕: 이상치를 무시할지(작은 C) 받아안을지(큰 C) — 마진 대 위반", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig("/home/smhan/book-ml/kor/src/images/ch05_2_C_sweep.svg")
plt.show()


## 4. 라벨 노이즈: $C$가 과적합을 조절하는 방식

실전 데이터의 더 흔한 문제는 "완벽한 이상치"가 아니라 **라벨링 오류**다.
클러스터 $\pm(3,3)$ (스프레드 1.5) 200개 중, $y{=}{+}1$ 클래스인데
경계 쪽에 가까운 30개 점의 라벨을 $-1$로 뒤집어 놓자(라벨 노이즈 15%).
이제 "올바른 답"은 라벨을 따라가면 **반대로** 성능이 떨어지는 답이다 —
$C$가 클수록 라벨(=오류)에 더 충실해지는 과적합이 시작된다.


In [5]:
from sklearn.linear_model import LogisticRegression

rng2 = np.random.default_rng(5)
n2 = 100
Xn = np.vstack([rng2.normal([3, 3], 1.5, (n2, 2)),
                rng2.normal([-3, -3], 1.5, (n2, 2))])
yn = np.r_[np.ones(n2), -np.ones(n2)]
pos_scores = Xn[:n2].sum(axis=1)          # 원점(경계)에 가까울수록 작음
flip_idx = np.argsort(pos_scores)[:30]    # +1 클러스터 중 경계 가장 가까운 30개
yn[flip_idx] = -1

rng3 = np.random.default_rng(77)
Xt = np.vstack([rng3.normal([3, 3], 1.5, (300, 2)),
                rng3.normal([-3, -3], 1.5, (300, 2))])
yt = np.r_[np.ones(300), -np.ones(300)]

print(f"학습 데이터: 200개 (이중 30개 라벨 오류),  테스트 데이터: 600개 (깨끗)")
print(f"{'모델':>10s} {'train':>8s} {'test':>8s} {'b/||w||':>9s}")
for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
    m = SVC(kernel="linear", C=C).fit(Xn, yn)
    wc, bc = m.coef_[0], m.intercept_[0]
    print(f"{'SVC C='+str(C):>10s} {m.score(Xn,yn):8.3f} {m.score(Xt,yt):8.3f} "
          f"{bc/np.linalg.norm(wc):+9.3f}")
m = LogisticRegression(max_iter=5000).fit(Xn, yn)
wc, bc = m.coef_[0], m.intercept_[0]
print(f"{'LogReg':>10s} {m.score(Xn,yn):8.3f} {m.score(Xt,yt):8.3f} {bc/np.linalg.norm(wc):+9.3f}")


학습 데이터: 200개 (이중 30개 라벨 오류),  테스트 데이터: 600개 (깨끗)
        모델    train     test   b/||w||
SVC C=0.01    0.865    0.968    -1.883
 SVC C=0.1    0.995    0.877    -3.131
 SVC C=1.0    0.995    0.870    -3.218
SVC C=10.0    0.995    0.875    -3.165
SVC C=100.0    1.000    0.877    -3.134
    LogReg    0.995    0.873    -3.172


In [6]:
# 결정 경계가 C에 따라 어떻게 "경계 쪽으로 밀리는지" 한 그림에
fig, ax = plt.subplots(figsize=(6.5, 6.5))
for lbl, m in [(1, "red"), (-1, "blue")]:
    ax.scatter(Xn[yn == lbl, 0], Xn[yn == lbl, 1], color=m, alpha=0.4, s=18)
ax.scatter(Xn[flip_idx, 0], Xn[flip_idx, 1], s=40, marker="x",
           color="red", lw=1.5, label="라벨 오류 30개")
x1 = np.linspace(-6.5, 6.5, 100)
for C, col in [(0.01, "#1a53ff"), (0.1, "#40c4ff"), (1.0, "#ffa500"),
               (10.0, "#ff6d00"), (100.0, "#ff1744")]:
    m = SVC(kernel="linear", C=C).fit(Xn, yn)
    wc, bc = m.coef_[0], m.intercept_[0]
    fl = lambda c: (-wc[0] * x1 - bc + c) / wc[1]
    ax.plot(x1, fl(0), color=col, lw=2, label=f"SVM C={C}")
m = LogisticRegression(max_iter=5000).fit(Xn, yn)
wc, bc = m.coef_[0], m.intercept_[0]
fl = lambda c: (-wc[0] * x1 - bc + c) / wc[1]
ax.plot(x1, fl(0), "purple", lw=2, ls=":", label="로지스틱회귀")
ax.set_aspect("equal"); ax.legend(fontsize=8, loc="lower right")
ax.set_title("라벨 노이즈 15%: C가 크면 경계가 오류 쪽으로 당겨짐")
fig.tight_layout()
fig.savefig("/home/smhan/book-ml/kor/src/images/ch05_2_margin_boundary.svg")
plt.show()


## 5. 힌지 vs 교차 엔트로피: "기여가 사라지는" 차이

5.1절 FAQ에서 예고한 차이 — 힌지 손실은 마진이 1을 넘으면 **점별 그래디언트
기여가 정확히 0**이 되지만, 교차 엔트로피는 아무리 확신 있게 맞혀도
$1-\sigma(f)>0$이므로 기여가 **절대 0이 안 된다**. 아래 표를 보자:
$y{=}{+}1$인 점이 기능적 마진 $f$일 때, 그 점이 $w$ 그래디언트에 기여하는
계수(\(y\,x\)의 크기 1 가정).


In [7]:
import math
def hinge_coef(f): return max(0.0, 1.0 - f)          # hinge: max(0,1-f)
def ce_coef(f):    return 1.0 / (1.0 + math.exp(f))  # CE: 1 - sigma(f) = sigma(-f)
print(f"{'f':>5s} {'힌지 기여':>10s} {'교차엔트로피 기여':>16s}")
for f in [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]:
    print(f"{f:5.1f} {hinge_coef(f):10.4f} {ce_coef(f):16.4f}")
print("\n-> f>=1이면 힌지는 정확히 0 (점 아예 무시), CE는 0으로 '수렴'만 함.")


    f      힌지 기여        교차엔트로피 기여
 -2.0     3.0000           0.8808
 -1.0     2.0000           0.7311
 -0.5     1.5000           0.6225
  0.0     1.0000           0.5000
  0.5     0.5000           0.3775
  1.0     0.0000           0.2689
  2.0     0.0000           0.1192
  3.0     0.0000           0.0474

-> f>=1이면 힌지는 정확히 0 (점 아예 무시), CE는 0으로 '수렴'만 함.


## 6. 정리: 다음으로

- **소프트 마진 = "제약 풀기 + 위반 페널티"**: $y(w^Tx+b)\ge1$을
  $y(w^Tx+b)\ge1-\xi^{(i)}$로 풀어놓고 $C\sum\xi$를 목적함수에 더함.
  $C$가 이 트레이드오프의 손잡이.
- **$C$는 항상 검증셋으로 튜닝**: $C\to\infty$는 하드 마진(이상치에 취약),
  $C\to0$은 아무것도 분류하지 않는 모델. 라벨 노이즈가 많을수록 작은 $C$.
- **힌지 손실의 "비활성화" 성질**이 서포트 벡터 개념을 가능하게 함.
- **5.3절 (커널 트릭)**: 선형분리 불가능한 데이터에서 커널로 고차원 내적을
  우회 — 소프트 마진 목적함수는 그대로, $w^Tx$가 커널로만 대체된다.
